In [4]:
"""
Results Analysis for Time Series Forecasting Models
Generates comparison tables for article publication
"""

import os
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD AND PREPARE DATA
# ==========================================

def load_and_prepare_results():
    """Load all result files and prepare consolidated dataframe"""
    
    df_list = []
    
    for root, dirs, files in os.walk('../results'):
        for f in files:
            if f.startswith('Result_') and f.endswith('.csv'):
                file_path = os.path.join(root, f)
                
                # Extract model and dataset from filename
                parts = f.replace('Result_', '').replace('.csv', '').split('_')
                
                if 'STL' in f or 'ASL' in f:
                    model = 'STL-ASL'
                    dataset_name = parts[-3] if len(parts) >= 3 else parts[0]
                elif 'MLP' in f:
                    model = 'MLP'
                    dataset_name = parts[1] if len(parts) > 1 else parts[0]
                elif 'SVR' in f:
                    model = 'SVR'
                    dataset_name = parts[1] if len(parts) > 1 else parts[0]
                elif 'ARIMA' in f:
                    model = 'ARIMA'
                    dataset_name = parts[1] if len(parts) > 1 else parts[0]
                elif 'LSTM' in f:
                    model = 'LSTM'
                    dataset_name = parts[1] if len(parts) > 1 else parts[0]
                else:
                    continue
                
                try:
                    df = pd.read_csv(file_path, sep=';', encoding='latin1', decimal='.')
                    
                    # Standardize column names
                    df.columns = [col.strip() for col in df.columns]
                    
                    # Add model and dataset columns
                    df['model'] = model
                    df['dataset'] = dataset_name
                    
                    # Clean dataset names for display
                    df['dataset_display'] = df['dataset'].replace({
                        'sunspot': 'Sunspot Numbers',
                        'canadian': 'Canadian Lynx',
                        'IBM': 'IBM Stock prices',
                        'TSLA': 'TSLA Stock prices'
                    })
                    
                    df_list.append(df)
                    
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    if not df_list:
        print("No result files found!")
        return pd.DataFrame()
    
    # Combine all dataframes
    df_all = pd.concat(df_list, ignore_index=True)
    
    # Ensure numeric columns
    numeric_cols = ['n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
    for col in numeric_cols:
        if col in df_all.columns:
            df_all[col] = pd.to_numeric(df_all[col], errors='coerce')
    
    return df_all


# ==========================================
# 2. TABLE 1: BEST RESULTS PER DATASET (ALL MODELS)
# ==========================================

def generate_table_best_per_dataset(df, output_file='table_best_per_dataset.csv'):
    """
    Generate table with best results for each dataset
    Sorted by sMAPE (lowest is best)
    """
    
    # Group by dataset and find row with minimum sMAPE
    idx = df.groupby(['dataset_display'])['sMAPE'].transform(min) == df['sMAPE']
    best_per_dataset = df[idx].copy()
    
    # Select and order columns
    result_df = best_per_dataset[[
        'dataset_display', 'model', 'n_time_steps', 
        'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration'
    ]].copy()
    
    # Round numeric columns
    result_df['MSE'] = result_df['MSE'].round(1)
    result_df['RMSE'] = result_df['RMSE'].round(1)
    result_df['MAE'] = result_df['MAE'].round(1)
    result_df['MAPE'] = result_df['MAPE'].round(1)
    result_df['sMAPE'] = result_df['sMAPE'].round(1)
    result_df['Duration'] = result_df['Duration'].round(1)
    
    # Sort by dataset name
    result_df = result_df.sort_values('dataset_display')
    
    # Rename columns for publication
    result_df.columns = [
        'Dataset', 'Model', 'n_time_steps', 
        'MSE', 'RMSE', 'MAE', 'MAPE (%)', 'sMAPE (%)', 'Duration (s)'
    ]
    
    # Save to CSV
    result_df.to_csv(output_file, sep=';', decimal=',', index=False)
    
    print(f"\n{'='*80}")
    print("TABLE 1: BEST RESULTS PER DATASET")
    print('='*80)
    print(result_df.to_string(index=False))
    
    return result_df


# ==========================================
# 3. TABLE 2: BEST RESULTS PER MODEL
# ==========================================

def generate_table_best_per_model(df, output_file='table_best_per_model.csv'):
    """
    Generate table with best results for each model across all datasets
    Sorted by sMAPE (lowest is best)
    """
    
    # Group by model and find row with minimum sMAPE
    idx = df.groupby(['model'])['sMAPE'].transform(min) == df['sMAPE']
    best_per_model = df[idx].copy()
    
    # Select and order columns
    result_df = best_per_model[[
        'model', 'dataset_display', 'n_time_steps',
        'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration'
    ]].copy()
    
    # Round numeric columns
    result_df['MSE'] = result_df['MSE'].round(1)
    result_df['RMSE'] = result_df['RMSE'].round(1)
    result_df['MAE'] = result_df['MAE'].round(1)
    result_df['MAPE'] = result_df['MAPE'].round(1)
    result_df['sMAPE'] = result_df['sMAPE'].round(1)
    result_df['Duration'] = result_df['Duration'].round(1)
    
    # Sort by sMAPE
    result_df = result_df.sort_values('sMAPE')
    
    # Rename columns for publication
    result_df.columns = [
        'Model', 'Dataset', 'n_time_steps',
        'MSE', 'RMSE', 'MAE', 'MAPE (%)', 'sMAPE (%)', 'Duration (s)'
    ]
    
    # Save to CSV
    result_df.to_csv(output_file, sep=';', decimal=',', index=False)
    
    print(f"\n{'='*80}")
    print("TABLE 2: BEST RESULTS PER MODEL")
    print('='*80)
    print(result_df.to_string(index=False))
    
    return result_df


# ==========================================
# 4. TABLE 3: COMPLETE RESULTS (ALL MODELS x ALL DATASETS)
# ==========================================

def generate_table_complete_results(df, output_file='table_complete_results.csv'):
    """
    Generate complete table with all model results for each dataset
    Sorted by dataset and then by sMAPE
    """
    
    # Select and order columns
    result_df = df[[
        'dataset_display', 'model', 'n_time_steps',
        'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration'
    ]].copy()
    
    # Round numeric columns
    result_df['MSE'] = result_df['MSE'].round(1)
    result_df['RMSE'] = result_df['RMSE'].round(1)
    result_df['MAE'] = result_df['MAE'].round(1)
    result_df['MAPE'] = result_df['MAPE'].round(1)
    result_df['sMAPE'] = result_df['sMAPE'].round(1)
    result_df['Duration'] = result_df['Duration'].round(1)
    
    # Sort by dataset and sMAPE
    result_df = result_df.sort_values(['dataset_display', 'sMAPE'])
    
    # Rename columns for publication
    result_df.columns = [
        'Dataset', 'Model', 'n_time_steps',
        'MSE', 'RMSE', 'MAE', 'MAPE (%)', 'sMAPE (%)', 'Duration (s)'
    ]
    
    # Save to CSV
    result_df.to_csv(output_file, sep=';', decimal=',', index=False)
    
    print(f"\n{'='*80}")
    print("TABLE 3: COMPLETE RESULTS (ALL MODELS)")
    print('='*80)
    print(result_df.to_string(index=False))
    
    return result_df


# ==========================================
# 5. TABLE 4: COMPARISON TABLE (LIKE IMAGE)
# ==========================================

def generate_table_comparison(df, output_file='table_comparison.csv'):
    """
    Generate comparison table similar to the image provided
    Shows best result per dataset with all models
    """
    
    # Get best result per dataset (lowest sMAPE)
    idx_best = df.groupby(['dataset_display'])['sMAPE'].transform(min) == df['sMAPE']
    best_results = df[idx_best][['dataset_display', 'model', 'sMAPE']].copy()
    best_results = best_results.set_index('dataset_display')['model'].to_dict()
    
    # Pivot table: datasets as rows, models as columns
    pivot_df = df.pivot_table(
        index='dataset_display',
        columns='model',
        values=['n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration'],
        aggfunc='first'
    )
    
    # Flatten column names
    pivot_df.columns = [f'{metric}_{model}' for metric, model in pivot_df.columns]
    pivot_df = pivot_df.reset_index()
    
    # Select and order columns for comparison
    models = ['STL-ASL', 'MLP', 'SVR', 'LSTM', 'ARIMA']
    metrics = ['sMAPE', 'MSE', 'RMSE', 'MAE', 'MAPE', 'Duration', 'n_time_steps']
    
    comparison_cols = ['dataset_display']
    for model in models:
        for metric in metrics:
            col_name = f'{metric}_{model}'
            if col_name in pivot_df.columns:
                comparison_cols.append(col_name)
    
    result_df = pivot_df[comparison_cols].copy()
    
    # Round numeric values
    for col in result_df.columns:
        if col != 'dataset_display':
            result_df[col] = pd.to_numeric(result_df[col], errors='coerce').round(1)
    
    # Convert ALL numeric columns to strings FIRST
    display_df = result_df.copy()
    for col in display_df.columns:
        if col != 'dataset_display':
            display_df[col] = display_df[col].astype(str)
    
    # Now add asterisks to the string columns
    for idx, row in result_df.iterrows():
        dataset = row['dataset_display']
        if dataset in best_results:
            best_model = best_results[dataset]
            col_name = f'sMAPE_{best_model}'
            if col_name in display_df.columns:
                # Get the original numeric value and add asterisk
                original_value = row[col_name]
                display_df.loc[idx, col_name] = f"{original_value}*"
    
    # Rename columns for publication
    rename_dict = {'dataset_display': 'Dataset'}
    for model in models:
        rename_dict[f'n_time_steps_{model}'] = f'{model} (steps)'
        rename_dict[f'MSE_{model}'] = f'{model} (MSE)'
        rename_dict[f'RMSE_{model}'] = f'{model} (RMSE)'
        rename_dict[f'MAE_{model}'] = f'{model} (MAE)'
        rename_dict[f'MAPE_{model}'] = f'{model} (MAPE %)'
        rename_dict[f'sMAPE_{model}'] = f'{model} (sMAPE %)'
        rename_dict[f'Duration_{model}'] = f'{model} (s)'
    
    display_df = display_df.rename(columns=rename_dict)
    
    # Save to CSV
    display_df.to_csv(output_file, sep=';', decimal=',', index=False)
    
    print(f"\n{'='*80}")
    print("TABLE 4: COMPARISON TABLE (ALL MODELS PER DATASET)")
    print('='*80)
    print(display_df.to_string(index=False))
    
    return display_df
# ==========================================
# 6. TABLE 5: SUMMARY STATISTICS
# ==========================================

def generate_table_summary(df, output_file='table_summary.csv'):
    """
    Generate summary statistics table
    """
    
    summary_data = []
    
    for dataset in df['dataset_display'].unique():
        dataset_df = df[df['dataset_display'] == dataset]
        
        for model in df['model'].unique():
            model_df = dataset_df[dataset_df['model'] == model]
            
            if len(model_df) > 0:
                # Get best result for this model on this dataset
                best_idx = model_df['sMAPE'].idxmin()
                best_row = model_df.loc[best_idx]
                
                summary_data.append({
                    'Dataset': dataset,
                    'Model': model,
                    'Best n_time_steps': int(best_row['n_time_steps']),
                    'Best sMAPE (%)': round(best_row['sMAPE'], 1),
                    'Best MAPE (%)': round(best_row['MAPE'], 1),
                    'MSE': round(best_row['MSE'], 1),
                    'RMSE': round(best_row['RMSE'], 1),
                    'MAE': round(best_row['MAE'], 1),
                    'Duration (s)': round(best_row['Duration'], 1)
                })
    
    result_df = pd.DataFrame(summary_data)
    result_df = result_df.sort_values(['Dataset', 'sMAPE (%)'])
    
    # Save to CSV
    result_df.to_csv(output_file, sep=';', decimal=',', index=False)
    
    print(f"\n{'='*80}")
    print("TABLE 5: SUMMARY STATISTICS")
    print('='*80)
    print(result_df.to_string(index=False))
    
    return result_df


# ==========================================
# 7. GENERATE LATEX TABLE FOR PUBLICATION
# ==========================================

def generate_latex_table(df, output_file='table_results.tex'):
    """
    Generate LaTeX table for academic publication
    """
    
    # Get best results per dataset
    idx = df.groupby(['dataset_display'])['sMAPE'].transform(min) == df['sMAPE']
    best_results = df[idx].copy()
    
    # Sort by dataset
    best_results = best_results.sort_values('dataset_display')
    
    # Prepare LaTeX content
    latex_lines = []
    latex_lines.append("\\begin{table}[htbp]")
    latex_lines.append("\\centering")
    latex_lines.append("\\caption{Best forecasting results per dataset}")
    latex_lines.append("\\label{tab:best_results}")
    latex_lines.append("\\begin{tabular}{lccccccccc}")
    latex_lines.append("\\hline")
    latex_lines.append("Dataset & Model & Steps & MSE & RMSE & MAE & MAPE (\\%) & sMAPE (\\%) & Time (s) \\\\")
    latex_lines.append("\\hline")
    
    for _, row in best_results.iterrows():
        dataset = row['dataset_display']
        model = row['model']
        steps = int(row['n_time_steps'])
        mse = round(row['MSE'], 2)
        rmse = round(row['RMSE'], 2)
        mae = round(row['MAE'], 2)
        mape = round(row['MAPE'], 2)
        smape = round(row['sMAPE'], 2)
        duration = round(row['Duration'], 2)
        
        line = f"{dataset} & {model} & {steps} & {mse:.2f} & {rmse:.2f} & {mae:.2f} & {mape:.2f} & {smape:.2f} & {duration:.2f} \\\\"
        latex_lines.append(line)
    
    latex_lines.append("\\hline")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{table}")
    
    with open(output_file, 'w') as f:
        f.write('\n'.join(latex_lines))
    
    print(f"\nLaTeX table saved to: {output_file}")
    
    return latex_lines


# ==========================================
# 8. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    
    print("="*80)
    print("TIME SERIES FORECASTING RESULTS ANALYSIS")
    print("="*80)
    
    # Load all results
    print("\nLoading result files...")
    df_results = load_and_prepare_results()
    
    if df_results.empty:
        print("No results found. Please run model scripts first.")
        sys.exit(1)
    
    print(f"Loaded {len(df_results)} result records")
    print(f"Models found: {df_results['model'].unique().tolist()}")
    print(f"Datasets found: {df_results['dataset_display'].unique().tolist()}")
    
    # Generate all tables
    print("\n" + "="*80)
    print("GENERATING TABLES")
    print("="*80)
    
    # Table 1: Best per dataset
    table1 = generate_table_best_per_dataset(df_results, 'table_best_per_dataset.csv')
    
    # Table 2: Best per model
    table2 = generate_table_best_per_model(df_results, 'table_best_per_model.csv')
    
    # Table 3: Complete results
    table3 = generate_table_complete_results(df_results, 'table_complete_results.csv')
    
    # Table 4: Comparison table (like image)
    table4 = generate_table_comparison(df_results, 'table_comparison.csv')
    
    # Table 5: Summary statistics
    table5 = generate_table_summary(df_results, 'table_summary.csv')
    
    # Generate LaTeX table for publication
    latex_table = generate_latex_table(df_results, 'table_results.tex')
    
    print("\n" + "="*80)
    print("ANALYSIS COMPLETED")
    print("="*80)
    print("\nGenerated files:")
    print("  - table_best_per_dataset.csv (Best results per dataset)")
    print("  - table_best_per_model.csv (Best results per model)")
    print("  - table_complete_results.csv (All results)")
    print("  - table_comparison.csv (Comparison table)")
    print("  - table_summary.csv (Summary statistics)")
    print("  - table_results.tex (LaTeX table for publication)")

TIME SERIES FORECASTING RESULTS ANALYSIS

Loading result files...
Loaded 195 result records
Models found: ['ARIMA', 'LSTM', 'MLP', 'STL-ASL', 'SVR']
Datasets found: ['Canadian Lynx', 'IBM Stock prices', 'Sunspot Numbers', 'TSLA Stock prices']

GENERATING TABLES

TABLE 1: BEST RESULTS PER DATASET
          Dataset Model  n_time_steps      MSE  RMSE   MAE  MAPE (%)  sMAPE (%)  Duration (s)
    Canadian Lynx   SVR             3 726960.0 852.6 356.2     111.9       25.3           1.6
 IBM Stock prices   SVR             1      2.4   1.5   0.9       1.0        0.5           1.6
 IBM Stock prices   SVR             2      2.3   1.5   0.9       1.0        0.5           1.6
 IBM Stock prices   SVR             5      2.3   1.5   0.9       1.0        0.5           1.5
 IBM Stock prices   SVR             6      2.4   1.5   0.9       1.0        0.5           1.6
TSLA Stock prices   SVR            14     71.4   8.5   6.3       3.2        1.6           1.7

TABLE 2: BEST RESULTS PER MODEL
  Model     

ValueError: Encountered all NA values